In [4]:
#%pip install --upgrade pip
#%pip install --upgrade optuna pandas numpy sqlalchemy joblib pathlib faker shap xgboost scikit-learn
# ---------------------------------------------------------
# 0. CREATE DIRECTORIES
#       a. Create directories
#       b. Initialize Faker components
# 1. DATA.ENV PARAMETERS LOAD
# 2. POSTGRESQL ENGINE
# 3. CORRELATION ANALYSIS
# 4. MAIN FUNCTION
#       a. Load CSV data, assess original shape
#       b. Add identity columns based on gender
#       c. Generate synthetic rows 
#       d. Run correlation analysis
#       e. Prepare df_extended for SQL write
#       f. PostgreSQL write
# ---------------------------------------------------------

# 0. IMPORT LIBRARIES AND CREATE DIRECTORIES
from pathlib import Path
from datetime import datetime
import os, random, urllib.parse, shap, joblib
import numpy as np
import pandas as pd
import sqlalchemy as sa
from sqlalchemy import text

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score
from xgboost import XGBClassifier

np.seterr(divide="ignore", invalid="ignore")
pd.options.display.max_columns = None
pd.set_option("display.max_rows", 100)

today = datetime.today().strftime("%Y-%m-%d")

# a. Create directories
os.makedirs("./Output", exist_ok=True)
os.makedirs("./Input", exist_ok=True)
os.makedirs("./model_save/model", exist_ok=True)

DATA_ENV_PATH = Path("./Input/data.env")
INPUT_CSV_PATH = Path("./Input/ibm_hr_synthetic_data.csv")

# b. Initialize Faker components
from faker import Faker
Faker.seed(42)
random.seed(42)
np.random.seed(42)

fake_sk = Faker("sk_SK")
fake_us = Faker("en_US")
fakers = [fake_sk, fake_us]

# 1. DATA.ENV PARAMETERS LOAD

def load_data_env(env_path: Path = DATA_ENV_PATH) -> None:
    if not env_path.exists():
        raise FileNotFoundError(f"Env file not found: {env_path}")

    with env_path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line or line.startswith("#"):
                continue
            if "=" not in line:
                continue
            key, value = line.split("=", 1)
            os.environ[key.strip()] = value.strip()


def env_params(key: str, default: str | None = None, required: bool = False) -> str | None:
    value = os.getenv(key, default)
    if required and not value:
        raise RuntimeError(f"Missing required env key: {key}")
    return value

# 2. POSTGRESQL ENGINE

def postgre_engine(env_path: Path = DATA_ENV_PATH) -> sa.Engine:
    load_data_env(env_path)

    pg_host = env_params("PG_HOST", required=True)
    pg_port = env_params("PG_PORT", "5432", required=True)
    pg_db = env_params("PG_DATABASE", required=True)
    pg_user = env_params("PG_USERNAME", required=True)
    pg_pwd = env_params("PG_PASSWORD", required=True)

    user_enc = urllib.parse.quote_plus(pg_user)
    pwd_enc = urllib.parse.quote_plus(pg_pwd)

    engine = sa.create_engine(f"postgresql+psycopg://{user_enc}:{pwd_enc}@{pg_host}:{pg_port}/{pg_db}",pool_pre_ping=True,future=True)
    return engine

def postgre_schema(env_path: Path = DATA_ENV_PATH) -> str:
    load_data_env(env_path)
    return env_params("PG_SCHEMA", "public") or "public"

def schema_exists(engine: sa.Engine, schema_name: str) -> None:
    with engine.begin() as conn:
        conn.execute(text(f'CREATE SCHEMA IF NOT EXISTS "{schema_name}"'))

def db_connection(engine: sa.Engine) -> None:
    with engine.connect() as conn:
        conn.execute(text("SELECT 1"))

def generate_identity(gender: str) -> pd.Series:
    faker = random.choice(fakers)

    if gender == "Male":
        first = faker.first_name_male()
    elif gender == "Female":
        first = faker.first_name_female()
    else:
        first = faker.first_name()

    if faker == fake_sk:
        if gender == "Male":
            last = faker.last_name_male()
        elif gender == "Female":
            last = faker.last_name_female()
        else:
            last = faker.last_name()
    else:
        last = faker.last_name()

    full = f"{first} {last}"
    email = f"{first.lower()}.{last.lower()}@sk_company.com"
    username = f"{first[0].lower()}{last.lower()}"

    return pd.Series(
        [first, last, full, email, username],
        index=["FirstName", "LastName", "FullName", "Email", "Username"])
    
# 3. CORRELATION ANALYSIS

def use_onehotencoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)


def build_attrition_analysis(df: pd.DataFrame,) -> tuple[pd.DataFrame, pd.Series]:
    if "Attrition" not in df.columns:
        raise ValueError("Column 'Attrition' not found in dataframe.")

    work_df = df.copy()

    analysis_drop_cols = ["EmployeeNumber",  "FirstName","LastName","FullName","Email","Username"]
    work_df = work_df.drop(columns=[c for c in analysis_drop_cols if c in work_df.columns], errors="ignore")

    # target
    y = work_df["Attrition"].map({"No": 0, "Yes": 1})
    if y.isna().any():
        raise ValueError("Column 'Attrition' contains unexpected values. Expected 'Yes'/'No'.")

    X = work_df.drop(columns=["Attrition"], errors="ignore")

    categorical_cols = X.select_dtypes(include=["object", "category", "string"]).columns.tolist()
    numeric_cols = X.select_dtypes(include=["number", "bool"]).columns.tolist()

    # numeric part
    X_num = X[numeric_cols].copy() if numeric_cols else pd.DataFrame(index=X.index)

    bool_cols = X_num.select_dtypes(include=["bool"]).columns.tolist()
    for col in bool_cols:
        X_num[col] = X_num[col].astype(int)

    # categorical part -> OHE
    if categorical_cols:
        ohe = use_onehotencoder()
        X_cat_encoded = ohe.fit_transform(X[categorical_cols].astype("string").fillna("__MISSING__"))
        X_cat_cols = ohe.get_feature_names_out(categorical_cols)
        X_cat = pd.DataFrame(X_cat_encoded, columns=X_cat_cols, index=X.index)
    else:
        X_cat = pd.DataFrame(index=X.index)

    X_final = pd.concat([X_num, X_cat], axis=1)

    # drop constant columns
    nunique = X_final.nunique(dropna=False)
    constant_cols = nunique[nunique <= 1].index.tolist()
    if constant_cols:
        X_final = X_final.drop(columns=constant_cols, errors="ignore")

    return X_final, y.astype(int)


def save_correlation(X: pd.DataFrame,y: pd.Series,output_dir: str | Path = "./Output",tag: str = today,) -> tuple[pd.DataFrame, Path, Path | None, Path | None]:

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    analysis_df = X.copy()
    analysis_df["Attrition"] = y.values

    corr_series = analysis_df.corr(numeric_only=True)["Attrition"].dropna()
    corr_series = corr_series.drop(labels=["Attrition"], errors="ignore").sort_values(ascending=False)

    corr_results = (corr_series.rename("CorrelationWithAttrition").reset_index().rename(columns={"index": "Feature"}))

    corr_csv_path = output_dir / f"attrition_correlation_ohe_{tag}.csv"
    corr_results.to_csv(corr_csv_path, index=False, encoding="utf-8-sig")

    heatmap_path = None
    top20_path = None

    if not corr_results.empty:
        heatmap_df = corr_results.set_index("Feature")[["CorrelationWithAttrition"]]

        plt.figure(figsize=(9, max(6, len(heatmap_df) * 0.30)))
        sns.heatmap(
                    heatmap_df,
                    annot=True,
                    fmt=".2f",
                    cmap="coolwarm",
                    center=0,
                    linewidths=0.3,
                    cbar=True
                    )
        plt.title("Correlation with Attrition (OneHotEncoder, analysis copy)")
        plt.xticks(rotation=0)
        plt.yticks(rotation=0)
        plt.tight_layout()

        heatmap_path = output_dir / f"attrition_correlation_heatmap_ohe_{tag}.png"
        plt.savefig(heatmap_path, dpi=200, bbox_inches="tight")
        plt.close()

        top_abs = corr_results.copy()
        top_abs["AbsCorrelation"] = top_abs["CorrelationWithAttrition"].abs()
        top_abs = top_abs.sort_values("AbsCorrelation", ascending=False).head(20)

        plt.figure(figsize=(11, max(6, len(top_abs) * 0.42)))
        sns.barplot(
            data=top_abs,
            x="CorrelationWithAttrition",
            y="Feature"
        )
        plt.title("Top 20 absolute correlations with Attrition")
        plt.tight_layout()

        top20_path = output_dir / f"attrition_top20_ohe_{tag}.png"
        plt.savefig(top20_path, dpi=200, bbox_inches="tight")
        plt.close()

    return corr_results, corr_csv_path, heatmap_path, top20_path


def save_feature_importance(X: pd.DataFrame,y: pd.Series,output_dir: str | Path = "./Output",tag: str = today,random_state: int = 42,) -> tuple[pd.DataFrame, Path, Path | None]:

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    model = RandomForestClassifier(n_estimators=300,random_state=random_state,n_jobs=-1,class_weight="balanced")
    model.fit(X, y)

    feature_df = pd.DataFrame({"Feature": X.columns,"Importance": model.feature_importances_}).sort_values("Importance", ascending=False)

    feature_csv_path = output_dir / f"attrition_feature_importance_{tag}.csv"
    feature_df.to_csv(feature_csv_path, index=False, encoding="utf-8-sig")

    feature_plot_path = None

    if not feature_df.empty:
        top_fi = feature_df.head(20).copy()

        plt.figure(figsize=(11, max(6, len(top_fi) * 0.42)))
        sns.barplot(
            data=top_fi,
            x="Importance",
            y="Feature"
        )
        plt.title("Top 20 Feature Importances for Attrition")
        plt.tight_layout()

        fi_plot_path = output_dir / f"attrition_feature_importance_top20_{tag}.png"
        plt.savefig(fi_plot_path, dpi=200, bbox_inches="tight")
        plt.close()

    return feature_df, feature_csv_path, feature_plot_path

def save_xgb_shap_analysis(X: pd.DataFrame,y: pd.Series,output_dir: str | Path = "./Output",tag: str = today,random_state: int = 42,shap_sample_size: int = 2000) -> tuple[pd.DataFrame, Path, Path | None, Path | None, Path | None]:

    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    X_train, X_test, y_train, y_test = train_test_split(X,y,test_size=0.20,random_state=random_state,stratify=y,)

    pos_count = int((y_train == 1).sum())
    neg_count = int((y_train == 0).sum())
    scale_pos_weight = (neg_count / pos_count) if pos_count > 0 else 1.0

    model = XGBClassifier(
                            n_estimators=900,
                            max_depth=13,
                            learning_rate=0.0002,
                            subsample=0.85,
                            colsample_bytree=0.92,
                            min_child_weight=5,
                            gamma=4.3,
                            reg_alpha=4.0,
                            reg_lambda=0.14,
                            random_state=random_state,
                            n_jobs=-1,
                            device="cuda",
                            tree_method="hist",
                            objective = "binary:logistic",
                            eval_metric=  "aucpr",
                            scale_pos_weight=scale_pos_weight,
                        )

    model.fit(X_train, y_train, verbose=False)

    y_prob = model.predict_proba(X_test)[:, 1]
    roc_auc = float(roc_auc_score(y_test, y_prob))
    pr_auc = float(average_precision_score(y_test, y_prob))

    metrics_df = pd.DataFrame(
        [
            {
                "Model": "XGBoost",
                "ROC_AUC": roc_auc,
                "PR_AUC": pr_auc,
                "TrainRows": len(X_train),
                "TestRows": len(X_test),
                "ScalePosWeight": scale_pos_weight,
            }
        ]
    )

    metrics_csv_path = output_dir / f"xgb_shap_metrics_{tag}.csv"
    metrics_df.to_csv(metrics_csv_path, index=False, encoding="utf-8-sig")

    model_path = output_dir / f"xgb_attrition_model_{tag}.pkl"
    joblib.dump(model, model_path)

    if len(X_train) > shap_sample_size:
        X_shap = X_train.sample(shap_sample_size, random_state=random_state).copy()
    else:
        X_shap = X_train.copy()

    explainer = shap.Explainer(model)
    shap_values = explainer(X_shap)

    shap_importance_df = pd.DataFrame(
        {
            "Feature": X_shap.columns,
            "MeanAbsSHAP": np.abs(shap_values.values).mean(axis=0),
        }
    ).sort_values("MeanAbsSHAP", ascending=False)

    shap_csv_path = output_dir / f"xgb_shap_importance_{tag}.csv"
    shap_importance_df.to_csv(shap_csv_path, index=False, encoding="utf-8-sig")

    beeswarm_path = None
    bar_path = None
    waterfall_path = None

    if not shap_importance_df.empty:
        # 1) Beeswarm
        try:
            plt.figure(figsize=(12, 8))
            shap.plots.beeswarm(shap_values, max_display=20, show=False)
            plt.title("SHAP Beeswarm - XGBoost Attrition")
            plt.tight_layout()
            beeswarm_path = output_dir / f"xgb_shap_beeswarm_{tag}.png"
            plt.savefig(beeswarm_path, dpi=200, bbox_inches="tight")
            plt.close()
        except Exception:
            try:
                plt.figure(figsize=(12, 8))
                shap.summary_plot(shap_values.values, X_shap, show=False, max_display=20)
                plt.title("SHAP Summary - XGBoost Attrition")
                plt.tight_layout()
                beeswarm_path = output_dir / f"xgb_shap_beeswarm_{tag}.png"
                plt.savefig(beeswarm_path, dpi=200, bbox_inches="tight")
                plt.close()
            except Exception:
                plt.close()

        # 2) Global bar
        try:
            plt.figure(figsize=(12, 8))
            shap.plots.bar(shap_values, max_display=20, show=False)
            plt.title("SHAP Bar - XGBoost Attrition")
            plt.tight_layout()
            bar_path = output_dir / f"xgb_shap_bar_{tag}.png"
            plt.savefig(bar_path, dpi=200, bbox_inches="tight")
            plt.close()
        except Exception:
            plt.close()

        # 3) Local waterfall for first sample
        try:
            plt.figure(figsize=(12, 8))
            shap.plots.waterfall(shap_values[0], max_display=20, show=False)
            plt.tight_layout()
            waterfall_path = output_dir / f"xgb_shap_waterfall_firstrow_{tag}.png"
            plt.savefig(waterfall_path, dpi=200, bbox_inches="tight")
            plt.close()
        except Exception:
            plt.close()

    return shap_importance_df, shap_csv_path, beeswarm_path, bar_path, waterfall_path

# 4. MAIN FUNCTION

def main() -> None:
    if not INPUT_CSV_PATH.exists():
        raise FileNotFoundError(f"Input CSV not found: {INPUT_CSV_PATH}")

    # a. Load CSV data
    df = pd.read_csv(INPUT_CSV_PATH)
    print("Original shape:", df.shape)

    # b. Add identity columns based on gender
    df[["FirstName", "LastName", "FullName", "Email", "Username"]] = df["Gender"].apply(generate_identity)

    cat_cols = [
                "Attrition",
                "Over18",
                "BusinessTravel",
                "Gender",
                "EducationField",
                "EnvironmentSatisfaction",
                "JobInvolvement",
                "JobLevel",
                "JobRole",
                "JobSatisfaction",
                "PerformanceRating",
                "RelationshipSatisfaction",
                "WorkLifeBalance",
                "MaritalStatus",
                "OverTime",
                "Department",
                "DistanceFromHome",
                "Education"
                ]

    sql_string_cols = cat_cols + [
                                    "FirstName",
                                    "LastName",
                                    "FullName",
                                    "Email",
                                    "Username"
                                ]

    cat_distributions = {
        col: df[col].value_counts(normalize=True)
        for col in cat_cols
        if col in df.columns
    }

    int_cols = df.select_dtypes(include=["int64", "int32"]).columns.tolist()
    float_cols = df.select_dtypes(include=["float64", "float32"]).columns.tolist()

    n_new = len(df) * 10
    synthetic_rows = []

    # c. Generate synthetic rows
    for _ in range(n_new):
        row = {}

        for col in cat_cols:
            if col not in cat_distributions:
                continue
            vals = cat_distributions[col].index
            probs = cat_distributions[col].values
            row[col] = np.random.choice(vals, p=probs)

        for col in int_cols:
            row[col] = int(np.random.choice(df[col].values))

        for col in float_cols:
            row[col] = float(np.random.choice(df[col].values))

        gender = row.get("Gender", None)
        faker = random.choice(fakers)

        if gender == "Male":
            first = faker.first_name_male()
        elif gender == "Female":
            first = faker.first_name_female()
        else:
            first = faker.first_name()

        if faker == fake_sk:
            if gender == "Male":
                last = faker.last_name_male()
            elif gender == "Female":
                last = faker.last_name_female()
            else:
                last = faker.last_name()
        else:
            last = faker.last_name()

        full = f"{first} {last}"
        email = f"{first.lower()}.{last.lower()}@sk_company.com"
        username = f"{first[0].lower()}{last.lower()}"

        row["FirstName"] = first
        row["LastName"] = last
        row["FullName"] = full
        row["Email"] = email
        row["Username"] = username

        synthetic_rows.append(row)

    df_new = pd.DataFrame(synthetic_rows)
    df_new = df_new.reindex(columns=df.columns)
    df_extended = pd.concat([df, df_new], ignore_index=True)

    if "EmployeeNumber" in df_extended.columns:
        df_extended["EmployeeNumber"] = range(1, len(df_extended) + 1)

    print("Extended shape:", df_extended.shape)
    print(df_extended.head(10))
    print(df_extended.info())

    # d. Run correlation analysis
    X_analysis, y_analysis = build_attrition_analysis(df_extended)

    corr_results, corr_csv_path, corr_heatmap_path, corr_top20_path = save_correlation(X_analysis,y_analysis,output_dir="./Output",tag=today)

    fi_results, fi_csv_path, fi_plot_path = save_feature_importance(X_analysis,y_analysis,output_dir="./Output",tag=today)
    
    shap_results, shap_csv_path, shap_beeswarm_path, shap_bar_path, shap_waterfall_path = save_xgb_shap_analysis(X_analysis,y_analysis,output_dir="./Output",tag=today,random_state=42,shap_sample_size=2000)

    print("\nAnalysis artifacts created")
    print(f"Correlation CSV: {corr_csv_path}")
    if corr_heatmap_path:
        print(f"Correlation heatmap: {corr_heatmap_path}")
    if corr_top20_path:
        print(f"Top 20 correlation chart: {corr_top20_path}")

    print(f"Feature importance CSV: {fi_csv_path}")
    if fi_plot_path:
        print(f"Feature importance chart: {fi_plot_path}")

    if not corr_results.empty:
        print("\nTop positive correlations with Attrition:")
        print(corr_results.sort_values("CorrelationWithAttrition", ascending=False).head(10).to_string(index=False))

        print("\nTop negative correlations with Attrition:")
        print(corr_results.sort_values("CorrelationWithAttrition", ascending=True).head(10).to_string(index=False))

    if not fi_results.empty:
        print("\nTop 15 feature importances:")
        print(fi_results.head(15).to_string(index=False))

    # e. Prepare df_extended for SQL write
    for c in sql_string_cols:
        if c in df_extended.columns:
            df_extended[c] = df_extended[c].astype("string")

    #f. SQL write
    engine = postgre_engine()
    schema_name = postgre_schema()
    table_name = "HR_Synth_Data"
    
    db_connection(engine)
    schema_exists(engine, schema_name)
    
    df_extended.to_sql(table_name,con=engine,if_exists="append",index=False,schema=schema_name,method="multi",chunksize=1000,)
    
    print(f"Data successfully written to {schema_name}.{table_name}")


if __name__ == "__main__":
    main()

Original shape: (1470, 35)
Extended shape: (16170, 40)
   Age Attrition     BusinessTravel  DailyRate              Department  \
0   41       Yes      Travel_Rarely       1102                   Sales   
1   49        No  Travel_Frequently        279  Research & Development   
2   37       Yes      Travel_Rarely       1373  Research & Development   
3   33        No  Travel_Frequently       1392  Research & Development   
4   27        No      Travel_Rarely        591  Research & Development   
5   32        No  Travel_Frequently       1005  Research & Development   
6   59        No      Travel_Rarely       1324  Research & Development   
7   30        No      Travel_Rarely       1358  Research & Development   
8   38        No  Travel_Frequently        216  Research & Development   
9   36        No      Travel_Rarely       1299  Research & Development   

   DistanceFromHome  Education EducationField  EmployeeCount  EmployeeNumber  \
0                 1          2  Life Sciences     

c:\Users\nyx3t\AppData\Local\Programs\Python\Python312\Lib\site-packages\xgboost\core.py:751: UserWarning: [13:59:06] WARNING: C:\actions-runner\_work\xgboost\xgboost\src\common\error_msg.cc:62: Falling back to prediction using DMatrix due to mismatched devices. This might lead to higher memory usage and slower performance. XGBoost is running on: cuda:0, while the input data is on: cpu.
Potential solutions:
- Use a data structure that matches the device ordinal in the booster.
- Set the device for booster before call to inplace_predict.

This warning will only be shown once.

  return func(**kwargs)



Analysis artifacts created
Correlation CSV: Output\attrition_correlation_ohe_2026-03-28.csv
Correlation heatmap: Output\attrition_correlation_heatmap_ohe_2026-03-28.png
Top 20 correlation chart: Output\attrition_top20_ohe_2026-03-28.png
Feature importance CSV: Output\attrition_feature_importance_2026-03-28.csv

Top positive correlations with Attrition:
                         Feature  CorrelationWithAttrition
                    OverTime_Yes                  0.030406
                Department_Sales                  0.019013
            MaritalStatus_Single                  0.018855
                DistanceFromHome                  0.018161
    JobRole_Sales Representative                  0.017451
   JobRole_Laboratory Technician                  0.013150
        EducationField_Marketing                  0.009466
      JobRole_Research Scientist                  0.009274
BusinessTravel_Travel_Frequently                  0.008559
              NumCompaniesWorked                  0.00

In [5]:
import torch

print(f"PyTorch version: {torch.__version__}")
print(f"Is CUDA available: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"CUDA version: {torch.version.cuda}")
    print(f"Current device: {torch.cuda.current_device()}")
    print(f"Device name: {torch.cuda.get_device_name(0)}")
    
    x = torch.tensor([1.0, 2.0]).to("cuda")
    print(f"Test tensor on GPU: {x}")
else:
    print("Check failed: PyTorch cannot access your GPU.")

PyTorch version: 2.11.0+cu126
Is CUDA available: True
CUDA version: 12.6
Current device: 0
Device name: NVIDIA GeForce RTX 4070 SUPER
Test tensor on GPU: tensor([1., 2.], device='cuda:0')
